# Predicting Loan Default: A Machine Learning Approach to Credit Risk Assessment

**Author:** Atilla Ahmed
**Dataset:** Lending Club Loan Data (2007–2018), ~2.26M loans, 145 features

---

## Table of Contents
1. [Setup and Imports](#1-setup-and-imports)
2. [Introduction and Problem Formulation](#2-introduction-and-problem-formulation)
3. [Data Loading and Initial Inspection](#3-data-loading-and-initial-inspection)
4. [Data Cleaning and Preprocessing](#4-data-cleaning-and-preprocessing)
5. [Exploratory Data Analysis](#5-exploratory-data-analysis)
6. [Feature Engineering & Preprocessing Pipeline](#6-feature-engineering)
7. [Mathematical Framework](#7-mathematical-framework)
8. [Baseline Models](#8-baseline-models)
9. [Advanced Models and Hyperparameter Tuning](#9-advanced-models-and-hyperparameter-tuning)
10. [Model Evaluation and Comparison](#10-model-evaluation-and-comparison)
11. [Business Impact Analysis](#11-business-impact-analysis)
12. [Conclusions, Limitations, and Future Work](#12-conclusions-limitations-and-future-work)
13. [References](#13-references)

## 1. Setup and Imports <a id="1-setup-and-imports"></a>

All libraries and global settings are defined in one place so the notebook is fully reproducible. The random seed is fixed at 42 throughout.

In [56]:
# SETUP — libraries, global config, random seed

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     RandomizedSearchCV, cross_val_score)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              confusion_matrix, classification_report,
                              roc_curve, precision_recall_curve)
from scipy.stats import randint, uniform

SEED = 42
np.random.seed(SEED)


## 2. Introduction and Problem Formulation <a id="2-introduction-and-problem-formulation"></a>

### 2.1 Background

Lending Club was the largest peer-to-peer lending platform in the US. Investors lend money directly to borrowers, and the central risk they face is credit default a borrower stops repaying. When that happens the investor loses part or all of their principal. Predicting which borrowers will default is therefore critical for anyone deploying capital on the platform.

This project builds a classification model to predict loan default using only information available at the time of loan issuance. No post-origination data is used.

### 2.2 Problem Statement

**Task:** Binary classification - *Fully Paid* (0) vs *Charged Off / Default* (1).

**Target:** `loan_status`, filtered to loans with a known final outcome only.

**Key constraint:** Only features available at origination. Any variable that reflects what happened during repayment is excluded using those would be data leakage and would produce a model that works perfectly in training but fails entirely on new applications.

**Evaluation metric:** Accuracy is misleading on imbalanced datasets. Missing a default costs real money; rejecting a good borrower costs opportunity. This asymmetry makes Recall and PR-AUC far more relevant.

### 2.3 Expected Loss Framework

$$\text{Expected Loss} = PD \times LGD \times EAD$$

- **PD (Probability of Default)** - the likelihood a borrower fails to repay. This is what the model predicts.
- **LGD (Loss Given Default)** - the fraction of the outstanding amount that is not recovered. Typically 40–60% for unsecured consumer loans.
- **EAD (Exposure at Default)** - the total amount owed at the time of default.

The model targets the PD component. A well-calibrated PD estimate allows a lender to price risk correctly and make informed accept/reject decisions on new applications.

## 3. Data Loading and Initial Inspection <a id="3-data-loading-and-initial-inspection"></a>

The dataset is a single 1.1 GB CSV file with 2.26 million loan records and 145 columns. A data dictionary is included as a separate Excel file.

In [57]:
loans = pd.read_csv('loan.csv', low_memory=False)
print(f'Shape: {loans.shape[0]:,} rows x {loans.shape[1]} columns')

Shape: 2,260,668 rows x 145 columns


In [58]:
loans.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,NaN,NaN,2500,2500,2500.0,36 months,13.56,84.92,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,30000,30000,30000.0,60 months,18.94,777.23,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,5000,5000,5000.0,36 months,17.97,180.69,D,D1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,4000,4000,4000.0,36 months,18.94,146.51,D,D2,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,30000,30000,30000.0,60 months,16.14,731.78,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
loans.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 145 columns):
 #    Column                                      Non-Null Count    Dtype  
---   ------                                      --------------    -----  
 0    id                                          0 non-null        float64
 1    member_id                                   0 non-null        float64
 2    loan_amnt                                   2260668 non-null  int64  
 3    funded_amnt                                 2260668 non-null  int64  
 4    funded_amnt_inv                             2260668 non-null  float64
 5    term                                        2260668 non-null  str    
 6    int_rate                                    2260668 non-null  float64
 7    installment                                 2260668 non-null  float64
 8    grade                                       2260668 non-null  str    
 9    sub_grade                                   2260668 non

### 3.1 Target Variable Distribution

Before doing anything else I need to understand what `loan_status` contains and decide which values map to default vs non-default.

In [60]:
(loans['loan_status'].value_counts(normalize=True).round(4) * 100)

loan_status
Fully Paid                                             46.09
Current                                                40.68
Charged Off                                            11.57
Late (31-120 days)                                      0.97
In Grace Period                                         0.40
Late (16-30 days)                                       0.17
Does not meet the credit policy. Status:Fully Paid      0.09
Does not meet the credit policy. Status:Charged Off     0.03
Default                                                 0.00
Name: proportion, dtype: float64

Looking at the distribution, a few things stand out:

- **Current (40.7%)** - these loans are still active. The outcome is unknown, so they have to be excluded.
- **Fully Paid (46.1%)** - borrower repaid in full → class 0.
- **Charged Off (11.6%)** - lender wrote off the debt → class 1.
- **Late / In Grace Period (1.5%)** - delinquent but not yet charged off. Ambiguous label, so excluded to avoid noise.
- The remaining categories (`Does not meet credit policy`, `Default`) are either negligible or will be mapped to their obvious outcome.

I keep only *Fully Paid* and *Charged Off* as the two definitive outcomes. Everything else is excluded.

### 3.2 Missing Values Overview

With 145 columns there will be significant gaps. This analysis is run on the raw dataset to get a baseline before any filtering. Worth noting: the effective missing rate for some columns will shift after I filter to known outcomes in Section 4, which is why I re-check missing values on the filtered data before deciding what to drop.

In [61]:
missing_raw = loans.isna().sum()
missing_pct_raw = (missing_raw / len(loans) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing' : missing_raw,
    'Percent' : missing_pct_raw
}).sort_values('Percent', ascending=False)

missing_df[missing_df['Missing'] > 0]

,Missing,Percent
id,2260668,100.00
url,2260668,100.00
member_id,2260668,100.00
orig_projected_additional_accrued_interest,2252242,99.63
hardship_length,2250055,99.53
...,...,...
acc_now_delinq,29,0.00
last_credit_pull_d,73,0.00
tax_liens,105,0.00
total_acc,29,0.00


Quick summary of how severe the missingness is:

In [62]:
total_cols = loans.shape[1]
empty_cols = (missing_pct_raw == 100).sum()
over_90 = (missing_pct_raw > 90).sum()
over_50 = (missing_pct_raw > 50).sum()

print(f'Total columns:         {total_cols}')
print(f'Completely empty:      {empty_cols}')
print(f'Over 90% missing:      {over_90}')
print(f'Over 50% missing:      {over_50}')

Total columns:         145
Completely empty:      3
Over 90% missing:      38
Over 50% missing:      44


Three columns are completely empty - `id`, `url`, `member_id` - stripped for privacy. The 38 columns above 90% missing are mostly hardship and settlement details that only exist for a small fraction of borrowers who entered those programs. The 44 columns above 50% include joint application fields populated only for co-borrowers, which is about 5% of loans.

Columns above 50% missing will be dropped, but the threshold will be re-evaluated after filtering.

## 4. Data Cleaning and Preprocessing <a id="4-data-cleaning-and-preprocessing"></a>

The raw dataset needs several rounds of cleaning before it is usable. I follow a deliberate order: filter first, then recheck missing rates, then drop, then convert types, then impute. Doing it in any other sequence can produce incorrect missing percentages or introduce subtle errors.

In [63]:
data_columns = loans.columns.to_list()

In [64]:
keep_status = [
    'Fully Paid',
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Fully Paid',
    'Does not meet the credit policy. Status:Charged Off'
]

loans = loans[loans['loan_status'].isin(keep_status)].copy()
print(f'Rows after filtering: {loans.shape[0]:,}')
print()
print(loans['loan_status'].value_counts())

Rows after filtering: 1,306,387

loan_status
Fully Paid                                             1041952
Charged Off                                             261655
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


### 4.2 Create Binary Target

In [65]:
default_statuses = [
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off'
]
loans['target'] = loans['loan_status'].isin(default_statuses).astype(int)

print(loans['target'].value_counts())
print()
print('Default rate:', f"{loans['target'].mean():.4f}")

target
0    1043940
1     262447
Name: count, dtype: int64

Default rate: 0.2009


### 4.3 Re-Compute Missing Rates on the Filtered Dataset

 Filtering toward known outcomes shifts the composition toward newer records, which can reduce apparent missingness. Re-computing on the 1.3M filtered rows ensures the 50% threshold reflects the data that will actually be used.

In [66]:
missing_filtered = loans.isna().sum()
missing_pct_filtered = (missing_filtered / len(loans) * 100).round(2)

missing_filtered_df = pd.DataFrame({
    'Missing' : missing_filtered,
    'Percent' : missing_pct_filtered
}).sort_values('Percent', ascending=False)

print('Columns with missing values after filtering:')
print(missing_filtered_df[missing_filtered_df['Missing'] > 0].to_string())

Columns with missing values after filtering:
                                            Missing  Percent
id                                          1306387   100.00
member_id                                   1306387   100.00
url                                         1306387   100.00
next_pymnt_d                                1303607    99.79
orig_projected_additional_accrued_interest  1302954    99.74
payment_plan_start_date                     1301052    99.59
hardship_type                               1301052    99.59
hardship_reason                             1301052    99.59
hardship_status                             1301052    99.59
deferral_term                               1301052    99.59
hardship_amount                             1301052    99.59
hardship_start_date                         1301052    99.59
hardship_end_date                           1301052    99.59
hardship_length                             1301052    99.59
hardship_dpd                            

### 4.4 Drop Unusable Columns

I split the columns to drop into four groups, each with a distinct reason:

**A) High-missing (>50% on filtered dataset)** - not enough data to impute reliably.

**B) Post-origination / leakage** - these variables only exist after a loan outcome has occurred. Using them would let the model see the future, producing inflated training metrics and a model that is completely useless on new applications.

**C) Zero-variance** - constant values carry no information and can destabilise some algorithms.

**D) Identifiers and free-text** - IDs, URLs, and unstructured text fields.

In [67]:
high_missing = missing_pct_filtered[missing_pct_filtered > 50].index.to_list()
print(f'A) High missing (>50%): {len(high_missing)} columns')

A) High missing (>50%): 58 columns


In [68]:
# Post-origination leakage none of these exist at loan origination
leakage_cols = [
    'out_prncp', 'out_prncp_inv',            # remaining principal — only known during repayment
    'total_pymnt', 'total_pymnt_inv',         # total payments received
    'total_rec_prncp', 'total_rec_int',       # principal and interest received
    'total_rec_late_fee',                      # late fees — only exist after missed payments
    'recoveries', 'collection_recovery_fee',  # post charge-off recovery amounts
    'last_pymnt_d', 'last_pymnt_amnt',        # last payment date and amount
    'next_pymnt_d',                            # next scheduled payment
    'last_credit_pull_d',                      # last time LC pulled credit
    'loan_status',                             # original target column
    'hardship_flag',                           # whether borrower entered a hardship program
    'debt_settlement_flag',                    # whether borrower entered debt settlement
    'disbursement_method',                     # how the loan was paid out — post-approval
    'pymnt_plan',                              # active payment plan — only set after repayment difficulty
]
print(f'B) Post-origination / leakage: {len(leakage_cols)} columns')

B) Post-origination / leakage: 18 columns


In [69]:
# Zero-variance constant across the entire dataset, no predictive signal
zero_variance_cols = [
    'policy_code',        # always 1 in the public dataset
    'initial_list_status' # LC internal listing flag, negligible signal
]
print(f'C) Zero-variance: {len(zero_variance_cols)} columns')

C) Zero-variance: 2 columns


In [70]:
irrelevant_cols = [
    'id', 'member_id', 'url',   # identifiers, all empty
    'emp_title',                 # free text, 300K+ unique values
    'title',                     # free text loan title, redundant with purpose
    'desc',                      # free text description, 94% missing
    'zip_code',                  # partial 3-digit zip, too granular
    'issue_d',                   # loan issue date temporal leakage if used as feature
]
print(f'D) Identifiers / irrelevant: {len(irrelevant_cols)} columns')

D) Identifiers / irrelevant: 8 columns


In [71]:
drop_cols = list(set(high_missing + leakage_cols + zero_variance_cols + irrelevant_cols))
print(f'Total unique columns to drop: {len(drop_cols)}')

loans.drop(columns=[c for c in drop_cols if c in loans.columns], inplace=True)
print(f'Remaining: {loans.shape[1]} columns')

Total unique columns to drop: 81
Remaining: 65 columns


### 4.5 Data Type Conversions

Several columns are stored as strings but contain numeric or categorical information.

In [72]:
str_cols = loans.select_dtypes(include='object').columns.to_list()
print(f'String columns: {len(str_cols)}')
print()
for col in str_cols:
    n_unique = loans[col].nunique()
    sample = loans[col].dropna().iloc[0] if loans[col].notna().any() else 'N/A'
    print(f'  {col:30s} {n_unique:>6} unique   example: {sample}')

String columns: 10

  term                                2 unique   example:  36 months
  grade                               7 unique   example: D
  sub_grade                          35 unique   example: D5
  emp_length                         11 unique   example: 5 years
  home_ownership                      6 unique   example: MORTGAGE
  verification_status                 3 unique   example: Source Verified
  purpose                            14 unique   example: debt_consolidation
  addr_state                         51 unique   example: CA
  earliest_cr_line                  738 unique   example: Jan-2012
  application_type                    2 unique   example: Joint App


In [73]:
# term: " 36 months" → 36
loans['term'] = loans['term'].str.strip().str.replace(' months', '').astype(int)
print(loans['term'].value_counts())

term
36    991212
60    315175
Name: count, dtype: int64


In [74]:
# emp_length: "10+ years" → 10, "< 1 year" → 0
loans['emp_length'] = (
    loans['emp_length']
    .str.replace(r'\+?\s*years?', '', regex=True)
    .str.replace('< 1', '0')
    .str.strip()
)
loans['emp_length'] = pd.to_numeric(loans['emp_length'], errors='coerce')
print(f'emp_length missing: {loans["emp_length"].isnull().sum():,}')

emp_length missing: 75,491


In [75]:
# earliest_cr_line: "Jan-2012" → years of credit history
# A raw date isn't useful as a feature. The number of years since the oldest
# credit line is a direct signal longer history generally means lower risk.
loans['earliest_cr_line'] = pd.to_datetime(loans['earliest_cr_line'], format='%b-%Y')
loans['credit_history_years'] = (
    (pd.Timestamp('2018-12-01') - loans['earliest_cr_line']).dt.days / 365.25
).round(1)
loans.drop(columns='earliest_cr_line', inplace=True)
print(loans['credit_history_years'].describe())

count    1.306358e+06
mean     1.978993e+01
std      7.591897e+00
min      3.200000e+00
25%      1.450000e+01
50%      1.840000e+01
75%      2.370000e+01
max      8.470000e+01
Name: credit_history_years, dtype: float64


### 4.6 Convert Categorical Columns

The remaining string columns are categorical. Converting to `category` dtype reduces memory usage.

In [76]:
cat_cols = ['sub_grade', 'home_ownership', 'verification_status',
            'purpose', 'addr_state', 'application_type']

for col in cat_cols:
    if col in loans.columns:
        loans[col] = loans[col].astype('category')

print(f'Memory: {loans.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'Categorical columns: {loans.select_dtypes(include="category").columns.tolist()}')

Memory: 0.63 GB
Categorical columns: ['sub_grade', 'home_ownership', 'verification_status', 'purpose', 'addr_state', 'application_type']


### 4.7 Handle Missing Values

Not all missing values mean the same thing. `mths_since_last_delinq` is missing when the borrower was never delinquent filling that with the mean would destroy the signal. Each column is handled according to why the data is absent.

In [77]:
remaining_missing = loans.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)

print(f'Columns with remaining missing values: {len(remaining_missing)}')
print()
for col, count in remaining_missing.items():
    pct = count / len(loans) * 100
    print(f'  {col:35s} {count:>8,}  ({pct:.1f}%)')

Columns with remaining missing values: 50

  mths_since_recent_inq                172,301  (13.2%)
  num_tl_120dpd_2m                     118,320  (9.1%)
  mo_sin_old_il_acct                   107,044  (8.2%)
  emp_length                            75,491  (5.8%)
  pct_tl_nvr_dlq                        70,430  (5.4%)
  avg_cur_bal                           70,297  (5.4%)
  mo_sin_rcnt_rev_tl_op                 70,277  (5.4%)
  mo_sin_old_rev_tl_op                  70,277  (5.4%)
  num_rev_accts                         70,277  (5.4%)
  num_tl_30dpd                          70,276  (5.4%)
  num_rev_tl_bal_gt_0                   70,276  (5.4%)
  total_il_high_credit_limit            70,276  (5.4%)
  num_actv_rev_tl                       70,276  (5.4%)
  num_bc_tl                             70,276  (5.4%)
  mo_sin_rcnt_tl                        70,276  (5.4%)
  tot_hi_cred_lim                       70,276  (5.4%)
  num_tl_op_past_12m                    70,276  (5.4%)
  num_il_tl          

In [78]:
# Safety check should be empty if Section 4.4 ran correctly
still_high = [col for col in remaining_missing.index
              if remaining_missing[col] / len(loans) > 0.50]

if still_high:
    print(f'WARNING: {len(still_high)} columns still >50% missing:')
    print(still_high)
    loans.drop(columns=still_high, inplace=True)
else:
    print('No columns above 50% missing — drop in Section 4.4 was complete.')

No columns above 50% missing — drop in Section 4.4 was complete.


### 4.8 Impute Remaining Missing Values

Three imputation strategies depending on the reason for missingness:

**A) "Months since" columns**  missing means the event never happened. A binary flag column captures this information before imputing with the median.

**B) `emp_length`**  missing likely means unemployed or unreported. Same approach: flag then fill with median.

**C) Everything else**  median for numeric columns, mode for categorical.

In [79]:
# A) "Months since" columns missing = event never occurred
months_since_cols = [c for c in ['mths_since_recent_inq', 'mo_sin_old_il_acct',
                                   'mths_since_recent_bc']
                     if c in loans.columns]

for col in months_since_cols:
    flag_col = col + '_missing'
    median_val = loans[col].median()

    loans[flag_col] = loans[col].isnull().astype(int)
    loans[col] = loans[col].fillna(median_val)

    print(f'  {col}: flagged {loans[flag_col].sum():,} → median fill ({median_val:.0f})')

  mths_since_recent_inq: flagged 172,301 → median fill (5)
  mo_sin_old_il_acct: flagged 107,044 → median fill (129)
  mths_since_recent_bc: flagged 62,491 → median fill (13)


In [80]:
# B) emp_length
loans['emp_length_missing'] = loans['emp_length'].isnull().astype(int)
loans['emp_length'] = loans['emp_length'].fillna(loans['emp_length'].median())
print(f'emp_length: {loans["emp_length_missing"].sum():,} flagged')

emp_length: 75,491 flagged


In [81]:
# C) Remaining nulls  median for numeric, mode for categorical
remaining_nulls = loans.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]

for col in remaining_nulls.index:
    count_before = loans[col].isnull().sum()
    if loans[col].dtype.name == 'category':
        fill_val = loans[col].mode()[0]
        loans[col] = loans[col].fillna(fill_val)
        print(f'  {col}: {count_before:,} NaNs → mode "{fill_val}"')
    else:
        fill_val = loans[col].median()
        loans[col] = loans[col].fillna(fill_val)
        print(f'  {col}: {count_before:,} NaNs → median {fill_val:.2f}')

print(f'\nTotal NaNs remaining: {loans.isnull().sum().sum()}')

  annual_inc: 4 NaNs → median 65000.00
  dti: 312 NaNs → median 17.60
  delinq_2yrs: 29 NaNs → median 0.00
  inq_last_6mths: 30 NaNs → median 0.00
  open_acc: 29 NaNs → median 11.00
  pub_rec: 29 NaNs → median 0.00
  revol_util: 850 NaNs → median 52.30
  total_acc: 29 NaNs → median 23.00
  collections_12_mths_ex_med: 145 NaNs → median 0.00
  acc_now_delinq: 29 NaNs → median 0.00
  tot_coll_amt: 70,276 NaNs → median 0.00
  tot_cur_bal: 70,276 NaNs → median 80335.00
  total_rev_hi_lim: 70,276 NaNs → median 24000.00
  acc_open_past_24mths: 50,030 NaNs → median 4.00
  avg_cur_bal: 70,297 NaNs → median 7416.00
  bc_open_to_buy: 63,389 NaNs → median 4662.00
  bc_util: 64,136 NaNs → median 63.40
  chargeoff_within_12_mths: 145 NaNs → median 0.00
  delinq_amnt: 29 NaNs → median 0.00
  mo_sin_old_rev_tl_op: 70,277 NaNs → median 164.00
  mo_sin_rcnt_rev_tl_op: 70,277 NaNs → median 8.00
  mo_sin_rcnt_tl: 70,276 NaNs → median 5.00
  mort_acc: 50,030 NaNs → median 1.00
  num_accts_ever_120_pd: 70,2

In [82]:
loans.head(10)

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,target,credit_history_years,mths_since_recent_inq_missing,mo_sin_old_il_acct_missing,mths_since_recent_bc_missing,emp_length_missing
100,30000,30000,30000.0,36,22.35,1151.16,D,D5,5.0,MORTGAGE,...,527120.0,98453.0,28600.0,101984.0,0,6.9,0,0,0,0
152,40000,40000,40000.0,60,16.14,975.71,C,C4,0.0,MORTGAGE,...,344802.0,161720.0,45700.0,167965.0,0,9.5,0,0,0,0
170,20000,20000,20000.0,36,7.56,622.68,A,A3,10.0,MORTGAGE,...,622183.0,71569.0,85100.0,74833.0,0,19.8,0,0,0,0
186,4500,4500,4500.0,36,11.31,147.99,B,B3,10.0,RENT,...,53795.0,29137.0,15100.0,24595.0,0,15.0,0,0,0,0
215,8425,8425,8425.0,36,27.27,345.18,E,E5,3.0,MORTGAGE,...,768304.0,189194.0,45800.0,189054.0,0,21.2,0,0,0,0
269,20000,20000,20000.0,60,17.97,507.55,D,D1,4.0,RENT,...,72700.0,33356.0,64800.0,0.0,0,23.7,0,0,0,0
271,6600,6600,6325.0,36,11.31,217.05,B,B3,10.0,RENT,...,38607.0,26836.0,10600.0,28007.0,0,9.7,1,0,0,0
296,2500,2500,2475.0,36,13.56,84.92,C,C1,5.0,RENT,...,32582.0,18649.0,10500.0,22082.0,0,14.8,0,0,0,0
369,4000,4000,4000.0,36,17.97,144.55,D,D1,5.0,MORTGAGE,...,127200.0,46250.0,4200.0,59474.0,0,11.5,0,0,0,0
379,2700,2700,2675.0,36,8.19,84.85,A,A4,4.0,OWN,...,78017.0,75363.0,6000.0,70617.0,0,16.2,0,0,0,0


### 4.9 Cleaning Summary

In [83]:
print(f'Rows:         {loans.shape[0]:,}')
print(f'Columns:      {loans.shape[1]}')
print(f'NaNs:         {loans.isnull().sum().sum()}')
print(f'Target dist:  {loans["target"].value_counts().to_dict()}')
print(f'Default rate: {loans["target"].mean():.2%}')
print(f'\nDtypes:')
print(loans.dtypes.value_counts())

Rows:         1,306,387
Columns:      69
NaNs:         0
Target dist:  {0: 1043940, 1: 262447}
Default rate: 20.09%

Dtypes:
float64     53
int32        6
int64        3
str          1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


Starting from 2.26M rows and 145 columns, I ended up with 1,306,387 rows. The dropped columns fell into four clear groups: high-missing after filtering, post-origination leakage, zero-variance constants, and identifiers. Binary flags preserve the informative absence in `emp_length` and the `mths_since_*` columns. Everything else is imputed with median or mode. No NaNs remain.